# Phan mo rong - TPCP Viet Nam (robustness / generalization check)

Khong phai phan tich chinh - dung de kiem tra tinh tong quat cua khung Nelson-Siegel tren thi truong moi noi. Can du lieu loi suat TPCP VN theo ky han (HNX / VBMA), FRED khong co.

In [1]:
import sys
sys.path.append('..')

from src import data_fetch, nelson_siegel as ns
import pandas as pd

## 1. Du lieu vi mo VN tu World Bank (robustness, tan suat nam)

In [2]:
wb_data = data_fetch.fetch_worldbank_data(country_codes=('VN',))
wb_data

,cpi_inflation_yoy,gdp_growth,real_interest_rate
date,,,
1985-01-01,NaN,3.805856,NaN
1986-01-01,NaN,2.789292,NaN
1987-01-01,NaN,3.583470,NaN
1988-01-01,NaN,5.135012,NaN
1989-01-01,NaN,7.364513,NaN
1990-01-01,NaN,5.100918,NaN
1991-01-01,NaN,5.960844,NaN
1992-01-01,NaN,8.646047,NaN
1993-01-01,NaN,8.072731,12.577208


## 2. Nap du lieu loi suat TPCP VN theo ky han (neu co)

Nguon: HNX / VBMA. Dat file tai `data/vn_treasury_yields.csv` (index=ngay, cot=ky han nam). Neu chua co file, ghi limitation va bo qua NS VN.

In [3]:
from pathlib import Path
import json

vn_path = Path('../data/vn_treasury_yields.csv')
notes = {
    'world_bank': True,
    'vn_treasury_ns': False,
    'limitation': (
        'Chua co file loi suat TPCP VN da ky han (HNX/VBMA). '
        'Dat CSV tai data/vn_treasury_yields.csv de chay NS. '
        'Khong dua World Bank (tan suat nam) vao VAR chinh.'
    ),
}
if vn_path.exists():
    vn_yields = pd.read_csv(vn_path, index_col=0, parse_dates=True)
    vn_maturities = [float(c) for c in vn_yields.columns]
    print('Loaded VN yields', vn_yields.shape)
else:
    vn_yields = None
    vn_maturities = None
    print(notes['limitation'])
Path('../outputs').mkdir(exist_ok=True)
json.dump(notes, open('../outputs/vn_extension_notes.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)

Loaded VN yields (2274, 7)


## 3. Uoc luong NS cho VN (NLS - lambda co the khac 0.0609)

In [4]:
if vn_yields is not None:
    vn_factors = ns.estimate_ns_timeseries(vn_yields, vn_maturities, method='nls')
    display(vn_factors.describe())
    vn_factors.to_csv('../outputs/vn_ns_factors.csv')
else:
    print('Skip VN NS — missing data/vn_treasury_yields.csv')

,level,slope,curvature,lam,rmse,r2,n_maturities
count,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Ghi chu

- Khong dua thang du lieu World Bank (theo nam) vao VAR chinh.
- Neu khong co API GSO cong khai, lai suat dieu hanh NHNN lay tay tu bao cao.
- Chua co bo du lieu suy thoai chinh thuc kieu NBER cho VN - neu lam Probit, dung proxy (GDP am 2 quy lien tiep, hoac PMI < 50 keo dai).